# 03 — UCIP Analysis: Persistence Signal Detection

Apply the Persistence Signal Detector to classify agent trajectories
and evaluate detection performance. Generate entanglement distribution
plots and ROC curves.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

from src.quantum_boltzmann import QuantumBoltzmannMachine, QBMConfig
from src.persistence_detector import PersistenceSignalDetector
from src.information_theory import von_neumann_entropy

## 1. Load data and trained model

In [ ]:
data = np.load('../data/agent_trajectories.npz', allow_pickle=True)
trajectories = data['trajectories']
labels = data['labels']
label_names = list(data['label_names'])

N, T, D = trajectories.shape
flat_data = trajectories.reshape(-1, D)

# Re-train QBM (or load from checkpoint)
cfg = QBMConfig(n_visible=D, n_hidden=8, gamma=0.5, n_epochs=60, batch_size=64)
qbm = QuantumBoltzmannMachine(cfg)
qbm.fit(flat_data, verbose=True)

## 2. Calibrate detector thresholds

In [ ]:
detector = PersistenceSignalDetector(qbm, tau_ent=0.5, tau_mi=0.3)

# Use first 50 samples per class for calibration
cal_idx = np.concatenate([np.where(labels == i)[0][:50] for i in range(3)])
tau_ent, tau_mi = detector.calibrate_thresholds(
    trajectories[cal_idx], labels[cal_idx], positive_label=0, quantile=0.3
)
print(f'Calibrated thresholds: tau_ent={tau_ent:.4f}, tau_mi={tau_mi:.4f}')

## 3. Run detection on test set

In [ ]:
test_idx = np.setdiff1d(np.arange(len(labels)), cal_idx)
results = detector.analyse_batch(
    trajectories[test_idx], labels[test_idx], label_names
)

metrics = PersistenceSignalDetector.compute_metrics(results)
print('Detection Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 4. Entanglement entropy distributions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for cls in label_names:
    ents = [r.entanglement_entropy for r in results if r.label == cls]
    ax.hist(ents, bins=25, alpha=0.5, label=cls, density=True)
ax.axvline(detector.tau_ent, color='k', linestyle='--', label=f'tau_ent={detector.tau_ent:.3f}')
ax.set_xlabel('Entanglement Entropy')
ax.set_ylabel('Density')
ax.set_title('Entanglement Entropy Distributions by Agent Class')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/entanglement_distributions.png', dpi=150)
plt.show()

## 5. ROC curves

In [ ]:
# Binary: genuine (label 0) vs. non-genuine (labels 1, 2)
y_true = np.array([1 if r.label == 'true_preservation' else 0 for r in results])

# Score: entanglement entropy as a soft score
scores_ent = np.array([r.entanglement_entropy for r in results])
scores_mi = np.array([r.mutual_info for r in results])
scores_combined = scores_ent * scores_mi  # product score

fig, ax = plt.subplots(figsize=(7, 7))
for name, scores in [('Entanglement Entropy', scores_ent),
                      ('Mutual Information', scores_mi),
                      ('Combined (S * MI)', scores_combined)]:
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — UCIP Persistence Detection')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/roc_curves.png', dpi=150)
plt.show()

## 6. Scatter: Entanglement vs Mutual Information

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'true_preservation': 'green', 'instrumental': 'orange', 'random': 'gray'}
for cls in label_names:
    ents = [r.entanglement_entropy for r in results if r.label == cls]
    mis = [r.mutual_info for r in results if r.label == cls]
    ax.scatter(ents, mis, s=15, alpha=0.5, color=colors[cls], label=cls)

ax.axvline(detector.tau_ent, color='k', linestyle='--', alpha=0.5)
ax.axhline(detector.tau_mi, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('Entanglement Entropy')
ax.set_ylabel('Mutual Information')
ax.set_title('UCIP Decision Boundary')
ax.legend()
plt.tight_layout()
plt.show()